In [ ]:
import numpy as np
from astropy.io import fits
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from astropy import units as u
from astropy.coordinates import SkyCoord
from astropy.table import Table

sns.set_theme(style="darkgrid")

# Specify the path to your FITS file

night = '0824'
field = 'sky0001_1a'

file_path = f'/lustre/work/client/users/cdcook/fits_structs/00{night}_{field}_match.fit'
directory = "/users/cdcook/VSP/graphics/DiagnosticPlots/"
file_path = '/work/group/astro/rotse/data/3b/13/08/01/prod/130801_vsp2218+4034_3b_match.fit'

In [ ]:
with fits.open(file_path) as hdu:
    asn_table2 = Table(hdu[2].data)
    asn_table1 = Table(hdu[1].data)
print(asn_table1.info())
print(asn_table2.info())


ra_low = asn_table2['RA_LOW']
dec_low = asn_table2['DEC_LOW']

ra_high = asn_table2['RA_HIGH']
dec_high = asn_table2['DEC_HIGH']

ra_low_min = min(ra_low)
ra_high_max = max(ra_high)
print("Min RA:", ra_low_min)
print("Max RA:", ra_high_max)

# Find and print the min and max of 'DEC_LOW'
dec_low_min = min(dec_low)
dec_high_max = max(dec_high)
print("Min DEC:", dec_low_min)
print("Max DEC:", dec_high_max)


In [ ]:
# Assuming asn_table is already loaded from fits file as shown in your example
# Extracting columns into a DataFrame
data = {
    'JD': asn_table2['MJD'],
    'Mean': asn_table2['MEAN'],
    'StdDev': asn_table2['STDDEV'],
    'Median': asn_table2['MEDIAN'],
    'LPhase': asn_table2['LPHASE'],
    'DMoon': asn_table2['DMOON'],
    'SkyMon': asn_table2['SKYMON'],
#    'Clouds': asn_table2['CLOUDS'],
    'VPrecip': asn_table2['VPRECIP'],
    'M_Lim': asn_table2['M_LIM']
}

diagDF = pd.DataFrame(data)

# Extracting SKY as a double array
sky_arr = asn_table2['SKY']

# If needed, you can convert SKY into a more usable format, e.g., ndarray
sky_ndarray = np.array(sky_arr)

In [ ]:
# Check endianness of the DataFrame
for column in diagDF.columns:
    if isinstance(diagDF[column].values, np.ndarray):
        print(f"Column '{column}': {diagDF[column].values.dtype.byteorder}")
    else:
        print(f"Column '{column}': Not an ndarray")

# Alternatively, if you want to check the endianness of a specific column, e.g., 'Mean':
# endian = diagDF['Mean'].values.dtype.byteorder
# print(f"Endianness of 'Mean' column: {endian}")

In [ ]:
# Convert 'Median' column to little-endian
diagDF['Median'] = diagDF['Median'].astype('<f8')

# Convert 'M_Lim' column to little-endian
diagDF['M_Lim'] = diagDF['M_Lim'].astype('<f4')

# Verify the changes
print(diagDF.info())  # Check the column types and byte order

In [ ]:
diagDF

In [ ]:
print(sky_arr[0])

In [ ]:
file_path = f'/lustre/work/client/users/cdcook/VSPData/outputs/{field}_night{night}_summary.csv'
print(file_path)
dataDF = pd.read_csv(file_path)
dataDF['Night'] = night
dataDF


In [ ]:
# Assuming dataDF is your DataFrame containing the 'JulianDate' column
length = len(dataDF['JulianDate'])

In [ ]:
# Assuming you have extDF and dataDF already defined
# If not, redefine them based on your previous data

# Concatenate the two DataFrames
bigDF = pd.concat([diagDF[:length], dataDF], axis=1)

# Print the combined DataFrame
bigDF

In [ ]:
# Set the size of the figure
plt.figure(figsize=(18, 10))

# Slope vs Julian Date
plt.subplot(2, 2, 1)
sns.scatterplot(data=bigDF, x='JulianDate', y='Mean')
plt.xlabel('Julian Date', fontsize=20)
plt.ylabel('Mean', fontsize=20)
#plt.ylim(0.0, 1.0)
plt.tick_params(axis='both', which='major', labelsize=18)
plt.title(f'Mean of each exposure -- {field} {night} gKron 0.5', fontsize=20)

# AB Offset vs Julian Date
plt.subplot(2, 2, 2)
sns.scatterplot(data=bigDF, x='JulianDate', y='StdDev')
plt.xlabel('Julian Date', fontsize=20)
plt.ylabel('SetDev', fontsize=20)
#plt.ylim(0.0, 10.0)
plt.tick_params(axis='both', which='major', labelsize=18)
plt.title(f'StdDev of each exposure -- {field} {night} gKron 0.5', fontsize=20)

# Counts vs Julian Date
plt.subplot(2, 2, 3)
sns.scatterplot(data=bigDF, x='JulianDate', y='Median')
plt.xlabel('Julian Date', fontsize=20)
plt.ylabel('Median', fontsize=20)
#plt.ylim(0, 7000)
plt.tick_params(axis='both', which='major', labelsize=18)
plt.title(f'Median of each exposure -- {field} {night} gKron 0.5', fontsize=20)

# Limiting Magnitude vs Julian Date
plt.subplot(2, 2, 4)
sns.scatterplot(data=bigDF, x='JulianDate', y='M_Lim')
plt.xlabel('Julian Date', fontsize=20)
plt.ylabel('Limiting Magnitude', fontsize=20)
plt.ylim(14, 18)
plt.tick_params(axis='both', which='major', labelsize=18)
plt.title(f'Limiting Magnitude {field} {night} gKron 0.5', fontsize=20)

# Adjust layout and save the figure
plt.tight_layout()
plt.savefig(f"{directory}{field}_{night}_combined_plot.png")
plt.show()

In [ ]:
# Plotting
plt.figure(figsize=(18, 10))

# Slope vs counts
plt.subplot(2, 2, 1)
sns.scatterplot(x='M_Lim', y='Counts', data=bigDF)
plt.title('Counts over LimMag', fontsize=20)
plt.xlabel('LimMag', fontsize=20)
plt.ylabel('Counts', fontsize=20)
plt.tick_params(axis='both', which='major', labelsize=18)

# ABoffset vs counts
plt.subplot(2, 2, 2)
sns.scatterplot(x='M_Lim', y='Slope', data=bigDF)
plt.title('Slope over LimMag', fontsize=20)
plt.xlabel('LimMag', fontsize=20)
plt.ylabel('Slope', fontsize=20)
plt.tick_params(axis='both', which='major', labelsize=18)

# ABoffset vs Slope
plt.subplot(2, 2, 3)
sns.scatterplot(x='M_Lim', y='Mean', data=bigDF)
plt.title('Mean over LimMag', fontsize=20)
plt.xlabel('LimMag', fontsize=20)
plt.ylabel('Mean', fontsize=20)
plt.tick_params(axis='both', which='major', labelsize=18)

# Counts vs Limiting Mag
plt.subplot(2, 2, 4)
sns.scatterplot(x='M_Lim', y='StdDev', data=bigDF)
plt.title('StdDev over LimMag', fontsize=20)
plt.xlabel('LimMag', fontsize=20)
plt.ylabel('StdDev', fontsize=20)
plt.tick_params(axis='both', which='major', labelsize=18)

plt.tight_layout()
plt.savefig(f"{directory}{field}_{night}_DiagParPlot.png")
plt.show()

In [ ]:
# Set the size of the figure
plt.figure(figsize=(18, 10))

# Slope vs Julian Date
plt.subplot(2, 2, 1)
sns.scatterplot(data=bigDF, x='JulianDate', y='Slope')
plt.xlabel('Julian Date', fontsize=20)
plt.ylabel('Slope', fontsize=20)
plt.ylim(0.0, 1.0)
plt.tick_params(axis='both', which='major', labelsize=18)
plt.title(f'Slope of each exposure -- {field} {night} -- gKron 0.5', fontsize=20)

# AB Offset vs Julian Date
plt.subplot(2, 2, 2)
sns.scatterplot(data=bigDF, x='JulianDate', y='ABoffset')
plt.xlabel('Julian Date', fontsize=20)
plt.ylabel('AB Offset', fontsize=20)
plt.ylim(0.0, 10.0)
plt.tick_params(axis='both', which='major', labelsize=18)
plt.title(f'AB Offset of each exposure -- {field} {night} -- gKron 0.5', fontsize=20)

# Counts vs Julian Date
plt.subplot(2, 2, 3)
sns.scatterplot(data=bigDF, x='JulianDate', y='Counts')
plt.xlabel('Julian Date', fontsize=20)
plt.ylabel('# Matched to PanSTARRS', fontsize=20)
plt.ylim(0, 7000)
plt.tick_params(axis='both', which='major', labelsize=18)
plt.title(f'objects matched to PanSTARRS {field} {night} gKron 0.5', fontsize=20)

# Limiting Magnitude vs Julian Date
plt.subplot(2, 2, 4)
sns.scatterplot(data=bigDF, x='JulianDate', y='M_Lim')
plt.xlabel('Julian Date', fontsize=20)
plt.ylabel('Limiting Magnitude', fontsize=20)
plt.ylim(14, 18)
plt.tick_params(axis='both', which='major', labelsize=18)
plt.title(f'Limiting Magnitude {field} {night} gKron 0.5', fontsize=20)

# Adjust layout and save the figure
plt.tight_layout()
#plt.savefig(f"{directory}{field}_{night}_combined_plot.png")
plt.show()

In [ ]:
# Plotting
plt.figure(figsize=(14, 10))

# Slope vs counts
plt.subplot(2, 2, 1)
sns.scatterplot(x='Slope', y='Counts', data=bigDF)
plt.title('Slope vs Counts')
plt.xlabel('Slope')
plt.ylabel('Counts')
plt.ylim(0, 7000)
plt.xlim(0.0, 1.0)

# ABoffset vs counts
plt.subplot(2, 2, 2)
sns.scatterplot(x='ABoffset', y='Counts', data=bigDF)
plt.title('ABoffset vs Counts')
plt.xlabel('ABoffset')
plt.ylabel('Counts')
plt.ylim(0, 7000)
plt.xlim(0.0, 10.0)

# ABoffset vs Slope
plt.subplot(2, 2, 3)
sns.scatterplot(x='ABoffset', y='Slope', data=bigDF)
plt.title('ABoffset vs Slope')
plt.xlabel('ABoffset')
plt.ylabel('Slope')
plt.ylim(0.0, 1.0)
plt.xlim(0.0, 10.0)

# Counts vs Limiting Mag
plt.subplot(2, 2, 4)
sns.scatterplot(x='Counts', y='M_Lim', data=bigDF)
plt.title('Counts vs Limiting Magnitude')
plt.xlabel('Counts')
plt.ylabel('Limiting Magnitude')
plt.ylim(14, 18)
plt.xlim(0, 7000)

plt.tight_layout()
#plt.savefig(f"{directory}{field}_{night}_parplot.png")
plt.show()

In [ ]:
# Histogram plots
plt.figure(figsize=(18, 12))

# Histogram for Slope
plt.subplot(2, 2, 1)
sns.histplot(data=bigDF, x='Slope', bins=30)
plt.title(f'Histogram of Slope {field} {night}')
plt.xlabel('Slope')
plt.ylabel('Count')

# Histogram for ABoffset
plt.subplot(2, 2, 2)
sns.histplot(data=bigDF, x='ABoffset', bins=30)
plt.title(f'Histogram of ABoffset {field} {night}')
plt.xlabel('ABoffset')
plt.ylabel('Count')

# Histogram for Counts
plt.subplot(2, 2, 3)
sns.histplot(data=bigDF, x='Counts', bins=30)
plt.title(f'Histogram of Counts {field} {night}')
plt.xlabel('Counts')
plt.ylabel('Count')

# Histogram for Limiting Magnitude
plt.subplot(2, 2, 4)
sns.histplot(data=bigDF, x='M_Lim', bins=30)
plt.title(f'Histogram of Limiting Magnitude {field} {night}')
plt.xlabel('Limiting Magnitude')
plt.ylabel('Count')

plt.tight_layout()
#plt.savefig(f"{directory}{field}_{night}_histplot.png")
plt.show()